# BERT Multi-Field Fine-Tuning + FCNN Classifier
## Fine-tune BERT for 4 text fields + combine with numerical features

---
## 1. Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pickle
import warnings
import os
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Ti
Memory: 8.22 GB


---
## 2. Configuration

In [2]:
# Text fields to fine-tune
TEXT_FEATURES = ['full_text', 'source', 'user.location', 'user.description']

# Numerical features
NUMERICAL_FEATURES = ['user.statuses_count', 'user.favourites_count', 'user.listed_count']

# Model configuration
MODEL_NAME = 'bert-base-multilingual-cased'  # Multilingual BERT
MAX_LENGTH = 280  # Sequence length

# Fine-tuning configuration
BATCH_SIZE = 16
FINETUNE_EPOCHS = 3
FINETUNE_LR = 2e-5
WARMUP_STEPS = 500

# FCNN configuration
FCNN_EPOCHS = 10
FCNN_LR = 1e-3
FCNN_BATCH_SIZE = 32

# Paths
MODELS_DIR = 'bert_finetuned_models'
os.makedirs(MODELS_DIR, exist_ok=True)

print("Configuration:")
print(f"  Text features: {TEXT_FEATURES}")
print(f"  Numerical features: {NUMERICAL_FEATURES}")
print(f"  Model: {MODEL_NAME}")
print(f"  Models will be saved to: {MODELS_DIR}/")

Configuration:
  Text features: ['full_text', 'source', 'user.location', 'user.description']
  Numerical features: ['user.statuses_count', 'user.favourites_count', 'user.listed_count']
  Model: bert-base-multilingual-cased
  Models will be saved to: bert_finetuned_models/


---
## 3. Load Data

In [3]:
print("Loading data...")

X_train = pd.read_pickle('original_data/data_train.pkl')
X_kaggle = pd.read_pickle('original_data/data_kaggle.pkl')
y_train = pd.read_pickle('original_data/y_train.pkl')

with open('features.pkl', 'rb') as f:
    features_data = pickle.load(f)
    kaggle_ids = features_data['kaggle_ids']

print(f"✓ Data loaded")
print(f"  Train samples: {len(X_train)}")
print(f"  Kaggle samples: {len(X_kaggle)}")
print(f"\nClass distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Class {label}: {count} ({count/len(y_train)*100:.2f}%)")

Loading data...
✓ Data loaded
  Train samples: 154914
  Kaggle samples: 103380

Class distribution:
  Class 0: 82674 (53.37%)
  Class 1: 72240 (46.63%)


---
## 4. Dataset for Fine-Tuning Single Text Field

In [4]:
class SingleFieldDataset(Dataset):
    """Dataset for fine-tuning on a single text field"""
    def __init__(self, texts, tokenizer, max_length, labels=None):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labels = labels
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx]) if pd.notna(self.texts[idx]) else ""
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return item

print("✓ Dataset class defined")

✓ Dataset class defined


---
## 5. BERT Classification Model with Embedding Extraction

In [5]:
class BertClassifier(nn.Module):
    """BERT model for classification with embedding extraction"""
    def __init__(self, model_name, num_labels=2, dropout=0.3):
        super(BertClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
    
    def forward(self, input_ids, attention_mask, return_embedding=False):
        # Get BERT outputs
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Use [CLS] token representation as sentence embedding
        pooled_output = outputs.pooler_output  # [batch_size, hidden_size]
        
        if return_embedding:
            return pooled_output
        
        # Classification
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

print("✓ Model class defined")

✓ Model class defined


---
## 6. Fine-Tuning Functions

In [6]:
def train_epoch(model, data_loader, optimizer, scheduler, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    progress_bar = tqdm(data_loader, desc="Training")
    
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        
        loss = nn.CrossEntropyLoss()(logits, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item()
        
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{correct/total:.4f}'
        })
    
    return total_loss / len(data_loader), correct / total


def extract_embeddings(model, data_loader, device):
    """Extract embeddings from fine-tuned model"""
    model.eval()
    all_embeddings = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Extracting embeddings"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            embeddings = model(input_ids=input_ids, attention_mask=attention_mask, return_embedding=True)
            all_embeddings.append(embeddings.cpu().numpy())
    
    return np.vstack(all_embeddings)

print("✓ Training functions defined")

✓ Training functions defined


---
## 7. Fine-Tune BERT for Each Text Field

In [7]:
# Load tokenizer
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print(f"✓ Tokenizer loaded: {MODEL_NAME}\n")

# Store fine-tuned models and embeddings
finetuned_models = {}
train_embeddings = {}
kaggle_embeddings = {}

# Fine-tune for each text field
for field in TEXT_FEATURES:
    print("="*70)
    print(f"Fine-tuning BERT for field: {field}")
    print("="*70)
    
    # Prepare data
    train_texts = X_train[field].fillna('').tolist()
    kaggle_texts = X_kaggle[field].fillna('').tolist()
    
    # Create datasets
    train_dataset = SingleFieldDataset(train_texts, tokenizer, MAX_LENGTH, y_train)
    kaggle_dataset = SingleFieldDataset(kaggle_texts, tokenizer, MAX_LENGTH)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    kaggle_loader = DataLoader(kaggle_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Initialize model
    model = BertClassifier(MODEL_NAME).to(device)
    
    # Setup optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=FINETUNE_LR, weight_decay=0.01)
    total_steps = len(train_loader) * FINETUNE_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=total_steps
    )
    
    # Train
    best_acc = 0
    for epoch in range(FINETUNE_EPOCHS):
        print(f"\nEpoch {epoch + 1}/{FINETUNE_EPOCHS}")
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
        
        print(f"  Loss: {train_loss:.4f} | Accuracy: {train_acc*100:.2f}%")
        
        if train_acc > best_acc:
            best_acc = train_acc
            # Save model
            model_path = os.path.join(MODELS_DIR, f'bert_finetuned_{field.replace(".", "_")}.pth')
            torch.save(model.state_dict(), model_path)
            print(f"  ✓ New best! Model saved to {model_path}")
    
    # Extract embeddings
    print(f"\nExtracting embeddings for {field}...")
    
    # Reload best model
    model.load_state_dict(torch.load(model_path))
    
    train_emb = extract_embeddings(model, train_loader, device)
    kaggle_emb = extract_embeddings(model, kaggle_loader, device)
    
    train_embeddings[field] = train_emb
    kaggle_embeddings[field] = kaggle_emb
    finetuned_models[field] = model
    
    print(f"  Train embeddings: {train_emb.shape}")
    print(f"  Kaggle embeddings: {kaggle_emb.shape}")
    print(f"  ✓ Embeddings extracted\n")

print("\n" + "="*70)
print("✓ All fields fine-tuned!")
print("="*70)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

✓ Tokenizer loaded: bert-base-multilingual-cased

Fine-tuning BERT for field: full_text


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]


Epoch 1/3


Training: 100%|██████████| 9683/9683 [1:02:55<00:00,  2.56it/s, loss=0.9905, acc=0.6678]


  Loss: 0.6083 | Accuracy: 66.78%
  ✓ New best! Model saved to bert_finetuned_models/bert_finetuned_full_text.pth

Epoch 2/3


Training:  89%|████████▉ | 8604/9683 [56:07<07:02,  2.55it/s, loss=0.5735, acc=0.7140] 


AcceleratorError: CUDA error: unspecified launch failure
Search for `cudaErrorLaunchFailure' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


---
## 8. Save Embeddings

In [ ]:
# Save embeddings
with open('bert_finetuned_embeddings.pkl', 'wb') as f:
    pickle.dump({
        'train_embeddings': train_embeddings,
        'kaggle_embeddings': kaggle_embeddings
    }, f)

print("✓ Embeddings saved to 'bert_finetuned_embeddings.pkl'")

---
## 9. Prepare Combined Features (Embeddings + Numerical)

In [ ]:
print("Preparing combined features...\n")

# Concatenate all text embeddings
train_text_features = np.hstack([train_embeddings[field] for field in TEXT_FEATURES])
kaggle_text_features = np.hstack([kaggle_embeddings[field] for field in TEXT_FEATURES])

print(f"Text embeddings concatenated:")
print(f"  Train: {train_text_features.shape}")
print(f"  Kaggle: {kaggle_text_features.shape}")

# Extract numerical features
train_numerical = X_train[NUMERICAL_FEATURES].fillna(0).values
kaggle_numerical = X_kaggle[NUMERICAL_FEATURES].fillna(0).values

print(f"\nNumerical features:")
print(f"  Train: {train_numerical.shape}")
print(f"  Kaggle: {kaggle_numerical.shape}")

# Normalize numerical features
scaler = StandardScaler()
train_numerical_scaled = scaler.fit_transform(train_numerical)
kaggle_numerical_scaled = scaler.transform(kaggle_numerical)

# Combine text embeddings + numerical features
X_train_combined = np.hstack([train_text_features, train_numerical_scaled])
X_kaggle_combined = np.hstack([kaggle_text_features, kaggle_numerical_scaled])

print(f"\n✓ Combined features:")
print(f"  Train: {X_train_combined.shape}")
print(f"  Kaggle: {X_kaggle_combined.shape}")

# Save scaler
with open('numerical_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f"\n✓ Scaler saved to 'numerical_scaler.pkl'")

---
## 10. Split Data for FCNN Training

In [ ]:
# Split into train/validation
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_combined, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"Data split:")
print(f"  Train: {X_train_split.shape}")
print(f"  Validation: {X_val_split.shape}")

---
## 11. FCNN Model Definition

In [ ]:
class FCNN(nn.Module):
    """Fully Connected Neural Network for classification"""
    def __init__(self, input_dim, hidden_dims=[512, 256, 128], dropout=0.3):
        super(FCNN, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # Hidden layers
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 2))
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

# Initialize FCNN
input_dim = X_train_combined.shape[1]
fcnn_model = FCNN(input_dim).to(device)

print(f"✓ FCNN initialized")
print(f"  Input dimension: {input_dim}")
print(f"  Architecture:")
print(fcnn_model)

---
## 12. FCNN Dataset

In [ ]:
class FCNNDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y) if y is not None else None
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

# Create datasets
train_fcnn_dataset = FCNNDataset(X_train_split, y_train_split)
val_fcnn_dataset = FCNNDataset(X_val_split, y_val_split)
kaggle_fcnn_dataset = FCNNDataset(X_kaggle_combined)

# Create dataloaders
train_fcnn_loader = DataLoader(train_fcnn_dataset, batch_size=FCNN_BATCH_SIZE, shuffle=True)
val_fcnn_loader = DataLoader(val_fcnn_dataset, batch_size=FCNN_BATCH_SIZE, shuffle=False)
kaggle_fcnn_loader = DataLoader(kaggle_fcnn_dataset, batch_size=FCNN_BATCH_SIZE, shuffle=False)

print("✓ FCNN DataLoaders ready")

---
## 13. Train FCNN

In [ ]:
def train_fcnn_epoch(model, train_loader, val_loader, optimizer, criterion, device):
    """Train FCNN for one epoch"""
    # Training
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for X_batch, y_batch in tqdm(train_loader, desc="Training FCNN"):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        predictions = torch.argmax(outputs, dim=1)
        train_correct += (predictions == y_batch).sum().item()
        train_total += y_batch.size(0)
    
    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            val_loss += loss.item()
            predictions = torch.argmax(outputs, dim=1)
            val_correct += (predictions == y_batch).sum().item()
            val_total += y_batch.size(0)
    
    return (
        train_loss / len(train_loader),
        train_correct / train_total,
        val_loss / len(val_loader),
        val_correct / val_total
    )

# Setup optimizer and loss
optimizer = AdamW(fcnn_model.parameters(), lr=FCNN_LR)
criterion = nn.CrossEntropyLoss()

# Train
print("\n" + "="*70)
print("Training FCNN Classifier")
print("="*70)

best_val_acc = 0
for epoch in range(FCNN_EPOCHS):
    train_loss, train_acc, val_loss, val_acc = train_fcnn_epoch(
        fcnn_model, train_fcnn_loader, val_fcnn_loader, optimizer, criterion, device
    )
    
    print(f"\nEpoch {epoch + 1}/{FCNN_EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(fcnn_model.state_dict(), os.path.join(MODELS_DIR, 'fcnn_classifier_best.pth'))
        print(f"  ✓ New best validation accuracy! Model saved.")

print("\n" + "="*70)
print(f"✓ FCNN training complete!")
print(f"  Best validation accuracy: {best_val_acc*100:.2f}%")
print("="*70)

---
## 14. Generate Final Predictions

In [ ]:
# Load best FCNN model
fcnn_model.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'fcnn_classifier_best.pth')))
fcnn_model.eval()

# Generate predictions
predictions = []
with torch.no_grad():
    for X_batch in tqdm(kaggle_fcnn_loader, desc="Predicting"):
        X_batch = X_batch.to(device)
        outputs = fcnn_model(X_batch)
        preds = torch.argmax(outputs, dim=1)
        predictions.extend(preds.cpu().numpy())

predictions = np.array(predictions)

print(f"\n✓ Predictions generated")
print(f"\nPrediction distribution:")
unique, counts = np.unique(predictions, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Class {label}: {count} ({count/len(predictions)*100:.2f}%)")

---
## 15. Save Predictions

In [ ]:
# Create submission
submission = pd.DataFrame({
    'ID': kaggle_ids,
    'Prediction': predictions
})

submission.to_csv('bert_fcnn_multifield_predictions.csv', index=False)

print("\n✓ Predictions saved to 'bert_fcnn_multifield_predictions.csv'")
print(f"\nSubmission preview:")
print(submission.head(10))
print(f"\nTotal predictions: {len(submission)}")

---
## 16. Summary

In [ ]:
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print("\nFine-tuned BERT models saved to:")
for field in TEXT_FEATURES:
    model_name = f'bert_finetuned_{field.replace(".", "_")}.pth'
    print(f"  - {MODELS_DIR}/{model_name}")

print(f"\nFCNN classifier saved to:")
print(f"  - {MODELS_DIR}/fcnn_classifier_best.pth")

print(f"\nEmbeddings saved to:")
print(f"  - bert_finetuned_embeddings.pkl")

print(f"\nScaler saved to:")
print(f"  - numerical_scaler.pkl")

print(f"\nPredictions saved to:")
print(f"  - bert_fcnn_multifield_predictions.csv")

print("\n" + "="*70)
print("✓ All done!")
print("="*70)